# Automated Tumor Detection in Whole Slide Images: An End-to-End Deep Learning Pipeline

**A practical guide to building a supervised deep learning system for detecting breast cancer metastases in histopathology images, using the CAMELYON16 challenge dataset.**

---

## 1. The Problem: Pathologist Shortages and the Promise of Automation

Diagnosing cancer from tissue biopsies remains one of the most critical — and most labour-intensive — tasks in modern medicine. A pathologist examining a sentinel lymph node biopsy must scan an entire tissue section at high magnification, searching for clusters of cancer cells that may occupy only a tiny fraction of the slide. In busy clinical settings, a single pathologist may review hundreds of slides per day, under significant time pressure.

The numbers tell a sobering story. In the UK, the Royal College of Pathologists has warned of a [sustained workforce crisis](https://www.rcpath.org/about-the-college/what-we-do/workforce-planning.html), with vacancy rates exceeding 30% in some specialties. In the US, the situation is similar: an ageing workforce, rising case volumes, and growing molecular testing demands all strain an already stretched system.

Meanwhile, the digitisation of histopathology is accelerating. Whole Slide Imaging (WSI) scanners now capture tissue sections at resolutions exceeding 100,000 × 100,000 pixels — gigapixel images that capture cellular-level detail across an entire tissue section. This creates an opportunity: if we can train machine learning models to analyse these images, we can augment pathologists' workflows, flag suspicious regions for closer review, and potentially catch metastases that might be missed under time pressure.

But there's a more ambitious hypothesis too. If tumors alter the surrounding tissue microenvironment — a phenomenon known as **field cancerization** — then machine learning might detect subtle changes in normal-appearing tissue that even expert pathologists would miss. This would represent a qualitatively different capability: not just automating what humans already do, but finding signals that humans cannot perceive.

In this post, we'll build an end-to-end pipeline that tests both of these ideas, from raw gigapixel images to trained classifiers, and we'll be honest about what works and what doesn't.


## 2. The CAMELYON16 Challenge

The [CAMELYON16 Grand Challenge](https://camelyon16.grand-challenge.org/) was organised in 2015–2016 by the International Symposium on Biomedical Imaging (ISBI) to benchmark automated detection of breast cancer metastases in whole slide images of sentinel lymph node biopsies.

The dataset consists of **400 H&E-stained whole slide images** from two Dutch medical centres (Radboud UMC and University Medical Centre Utrecht):

| Split | Tumor slides | Normal slides | Total |
|-------|-------------|--------------|-------|
| Train | 110 | 160 | 270 |
| Test  | 49  | 80  | 129 |

Each tumor slide comes with XML annotation files containing polygon outlines of metastatic regions, hand-drawn by expert pathologists.

The winning team (Wang et al., 2016) achieved a slide-level AUC of 0.925 using a GoogLeNet-based patch classifier trained on millions of patches. Remarkably, when combined with a pathologist's review, the system reduced the pathologist's error rate from 3.4% to under 1%.

Our approach follows the same fundamental strategy — patch-based classification — but with a twist: we introduce a **four-class labelling scheme** that lets us ask more nuanced questions about what the model is actually detecting.


## 3. Environment Setup

This project runs on **Google Colab** with a GPU runtime. The codebase is modular: data utilities, model architectures, and training logic are separated into clean Python modules.

**Sections 3–6** (data exploration, tissue detection, visualisation) are fully runnable by anyone — they download individual slides from S3 on the fly.

**Sections 10–12** (model training and evaluation) require the pre-generated patch dataset. You can either generate it yourself using the code in Section 7 (~6–8 hours on Colab), or adapt the dataset paths to point to your own copy.

> **Note**: Model training requires Colab Pro (High RAM) to avoid out-of-memory crashes during the `tf.data` pipeline. All other sections run on the free tier.


In [ ]:
# Mount Google Drive and navigate to project folder
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

import numpy as np
import matplotlib.pyplot as plt
import openslide

np.random.seed(42)

# Project imports
from config import DEFAULT_CONFIG
from src.data import list_s3_files, download_file_from_s3, cleanup_file
from src.data.tissue_mask import get_tissue_mask, compute_foreground_mask
from src.data.tumor_polygons import load_tumor_polygons, classify_patch
from src.data.patch_extraction import (
    sample_grid_coordinates, sample_coordinates_by_class,
    extract_patch, preprocess_patch
)
from src.visualisation import (
    visualise_tissue_outline, visualise_patches_grid,
    find_zoom_region_by_coords, find_dense_tissue_region
)

print("All imports successful!")

## 4. Understanding Whole Slide Images

A WSI is not a regular image. At full resolution, a single slide can be 100,000 × 200,000 pixels — that's roughly **60 gigabytes** of uncompressed pixel data. You cannot load one into memory.

Instead, WSI formats (like TIFF) use a **pyramidal structure**: the same image stored at multiple resolutions. We use the [OpenSlide](https://openslide.org/) library to navigate this pyramid, reading small regions on demand without loading the entire file.

Let's download a single slide and explore its structure.


In [ ]:
# List available slides from S3
from config import DEFAULT_CONFIG

all_slides = list_s3_files(DEFAULT_CONFIG.data.s3_images, '.tif')
normal_slides = sorted([f for f in all_slides if 'normal' in f.lower()])
tumor_slides = sorted([f for f in all_slides if 'tumor' in f.lower()])

print(f"Dataset: {len(normal_slides)} normal slides, {len(tumor_slides)} tumor slides")
print(f"\nExample normal slide: {normal_slides[0]}")
print(f"Example tumor slide: {tumor_slides[0]}")


In [ ]:
# Download one tumor slide to explore
slide_name = 'tumor_010.tif'
slide_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_images + slide_name,
    f'/tmp/{slide_name}'
)
slide = openslide.OpenSlide(slide_path)

# Explore the pyramid structure
print(f"Slide: {slide_name}")
print(f"Dimensions (level 0): {slide.dimensions[0]:,} × {slide.dimensions[1]:,} pixels")
print(f"Number of levels: {slide.level_count}")
print(f"\nPyramid levels:")
for i in range(slide.level_count):
    w, h = slide.level_dimensions[i]
    ds = slide.level_downsamples[i]
    print(f"  Level {i}: {w:>7,} × {h:>7,}  (downsample: {ds:.1f}×)")


In [ ]:
# View the whole slide as a thumbnail
thumbnail = slide.get_thumbnail((800, 800))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(thumbnail)
ax.set_title(f'{slide_name} — Thumbnail', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nThe thumbnail is {thumbnail.size[0]}×{thumbnail.size[1]} pixels.")
print(f"The actual slide is {slide.dimensions[0]:,}×{slide.dimensions[1]:,} pixels.")
print(f"That's a {slide.dimensions[0] // thumbnail.size[0]:,}× reduction!")


## 5. Step 1 — Finding the Tissue

Most of a WSI is empty white background (the glass slide). Before extracting patches, we need to find where the actual tissue is. This is a simple but crucial preprocessing step.

Our approach works at **thumbnail resolution** for speed:
1. Convert to grayscale (tissue is darker than background)
2. Apply a brightness threshold
3. Clean up with morphological operations (remove dust specks, fill holes)

This takes less than a second per slide — versus the hours it would take to scan the full-resolution image.


### Under the Hood: Tissue Masking

The tissue masking logic is surprisingly simple. Here's what `compute_foreground_mask` does — just ~10 lines of core logic:


In [ ]:
# === What compute_foreground_mask does internally ===
# (This is the actual logic — we'll use the module version below for the pipeline)

from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.segmentation import clear_border

# Step 1: Get a tiny thumbnail (512×512) from the gigapixel image
thumbnail_gray = slide.get_thumbnail((512, 512)).convert("L")
thumbnail_array = np.array(thumbnail_gray)

# Step 2: Simple brightness threshold
# Tissue is darker than the white glass background
threshold = 220  # pixels darker than this are tissue
raw_mask = thumbnail_array < threshold

# Step 3: Morphological cleanup
cleaned = remove_small_objects(raw_mask, min_size=500)   # remove dust specks
cleaned = clear_border(cleaned)                           # remove edge artifacts
cleaned = remove_small_holes(cleaned, area_threshold=1000) # fill holes in tissue

print(f"Raw mask pixels:     {raw_mask.sum():,}")
print(f"After cleanup:       {cleaned.sum():,}")
print(f"Removed {raw_mask.sum() - cleaned.sum():,} artifact pixels")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(thumbnail_array, cmap='gray')
axes[0].set_title('Grayscale thumbnail')
axes[1].imshow(raw_mask, cmap='gray')
axes[1].set_title(f'After threshold (<{threshold})')
axes[2].imshow(cleaned, cmap='gray')
axes[2].set_title('After morphological cleanup')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Generate tissue mask
mask = compute_foreground_mask(slide)

print(f"Mask shape: {mask.shape}")
print(f"Tissue coverage: {mask.sum() / mask.size:.1%}")

# Visualise: original thumbnail vs. detected tissue
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original
axes[0].imshow(thumbnail)
axes[0].set_title('Original Slide', fontsize=13)
axes[0].axis('off')

# Binary mask
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Tissue Mask', fontsize=13)
axes[1].axis('off')

# Overlay: tissue outline on slide
axes[2].imshow(thumbnail)
# Resize mask to match thumbnail
from PIL import Image
mask_resized = np.array(
    Image.fromarray(mask.astype(np.uint8) * 255).resize(thumbnail.size, Image.NEAREST)
) > 127
axes[2].contour(mask_resized.astype(float), levels=[0.5], colors='lime', linewidths=1.5)
axes[2].set_title('Tissue Detection Overlay', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()


## 6. Step 2 — Parsing Tumor Annotations and the Four-Class Labelling Scheme

For tumor slides, CAMELYON16 provides XML files with polygon annotations tracing the tumor boundaries. We parse these into [Shapely](https://shapely.readthedocs.io/) geometry objects, which lets us compute precise overlap between any patch and the annotated tumor regions.

### The Four-Class Scheme

Rather than a simple binary "normal vs. tumor" label, we classify each patch into **four categories** based on its spatial relationship to the tumor:

| Class | Name | Description |
|-------|------|-------------|
| 0 | `normal_from_normal` | Normal tissue from a slide with no tumor at all |
| 1 | `normal_from_tumor` | Normal-looking tissue on a slide that also contains tumor |
| 2 | `boundary_tumor` | Tissue at the tumor margin (partial overlap with annotations) |
| 3 | `pure_tumor` | Tissue fully within annotated tumor regions |

This scheme lets us ask more interesting questions than simple binary classification. In particular, **Class 1 vs. Class 0** tests the field cancerization hypothesis: is there a detectable difference between "normal" tissue that happens to be near a tumor and normal tissue from a completely healthy slide?


In [ ]:
# Load tumor annotations
xml_path = download_file_from_s3(
    DEFAULT_CONFIG.data.s3_annotations + 'tumor_010.xml',
    '/tmp/tumor_010.xml'
)
polygons = load_tumor_polygons(xml_path)
print(f"Loaded {len(polygons)} tumor polygons")
for i, p in enumerate(polygons):
    print(f"  Polygon {i}: area = {p.area:,.0f} pixels², "
          f"bounds = {tuple(int(x) for x in p.bounds)}")


In [ ]:
# Visualise tumor annotations overlaid on the tissue
visualise_tissue_outline(
    slide, mask,
    tumor_polygons=polygons,
    title='Tumor Annotations (red) with Tissue Outline (green)',
    figsize=(10, 10)
)


### Classifying Patches by Tumor Overlap

Each patch is classified by computing the **fractional overlap** between the 224×224 patch and the tumor polygons:
- **< 1% overlap** → Class 1 (normal tissue on tumor slide)
- **1–50% overlap** → Class 2 (boundary tissue)
- **≥ 50% overlap** → Class 3 (pure tumor)

The thresholds are configurable, but these defaults ensure clean separation between classes.


### Under the Hood: Patch Classification with Shapely

How do we turn a patch coordinate into a class label? The key is computing the **geometric overlap** between the patch (a square) and the tumor polygons. Here's the core logic:


In [ ]:
# === What classify_patch does internally ===
from shapely.geometry import Polygon, box

# Pick an example coordinate near a tumor boundary
example_coords = coords_by_class_preview = sample_coordinates_by_class(slide_path, xml_path)

# Show the classification logic for 3 example patches (one per class)
for class_id in [1, 2, 3]:
    coords = example_coords.get(class_id, [])
    if not coords:
        continue
    x, y = coords[0]  # Take first patch of this class

    # Create a square box for the patch
    patch_size = 224
    half = patch_size // 2
    patch_box = box(x - half, y - half, x + half, y + half)
    patch_area = patch_size * patch_size

    # Compute intersection with ALL tumor polygons
    total_overlap = 0.0
    for polygon in polygons:
        if polygon.intersects(patch_box):
            intersection = polygon.intersection(patch_box)
            total_overlap += intersection.area

    overlap_fraction = min(total_overlap / patch_area, 1.0)

    # Apply classification thresholds
    if overlap_fraction < 0.01:
        label_name = "Class 1: Normal (tumor slide)"
    elif overlap_fraction < 0.50:
        label_name = "Class 2: Boundary"
    else:
        label_name = "Class 3: Pure Tumor"

    print(f"  Patch at ({x}, {y}): overlap = {overlap_fraction:.1%} → {label_name}")


In [ ]:
# Sample coordinates by class from this tumor slide
from src.data.patch_extraction import sample_coordinates_by_class

coords_by_class = sample_coordinates_by_class(
    slide_path, xml_path
)

for class_id, coords in sorted(coords_by_class.items()):
    class_names = {1: 'Normal (tumor slide)', 2: 'Boundary', 3: 'Pure Tumor'}
    print(f"  Class {class_id} ({class_names[class_id]}): {len(coords)} patches")


In [ ]:
# Visualise patch grid zoomed into the tumor region
# Find a zoom region centered on the tumor
tumor_coords = coords_by_class.get(3, []) + coords_by_class.get(2, [])
if tumor_coords:
    zoom = find_zoom_region_by_coords(tumor_coords, region_size=10000)

    visualise_patches_grid(
        slide,
        coords_by_class={
            1: coords_by_class.get(1, []),
            2: coords_by_class.get(2, []),
            3: coords_by_class.get(3, [])
        },
        zoom_region=zoom,
        class_colours={1: 'blue', 2: 'orange', 3: 'red'},
        class_labels={
            1: 'Normal (tumor slide)',
            2: 'Boundary',
            3: 'Pure Tumor'
        },
        title=f'Patch Classification — {slide_name}',
        figsize=(14, 12)
    )


In [ ]:
# Show example patches from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

class_names = {1: 'Normal (tumor slide)', 2: 'Boundary', 3: 'Pure Tumor'}
class_colours = {1: 'steelblue', 2: 'darkorange', 3: 'firebrick'}

for col, (class_id, name) in enumerate(class_names.items()):
    coords = coords_by_class.get(class_id, [])
    if len(coords) < 2:
        continue

    for row in range(2):
        x, y = coords[row]
        patch = extract_patch(slide, x, y)
        axes[row, col].imshow(patch)
        axes[row, col].set_title(f'Class {class_id}: {name}', fontsize=10,
                                  color=class_colours[class_id], fontweight='bold')
        axes[row, col].axis('off')

# Also show a normal patch from a normal slide (class 0)
# We'll use a placeholder title for now
axes[0, 3].text(0.5, 0.5, 'Class 0:\nNormal from\nnormal slide\n(see Section 7)',
                ha='center', va='center', fontsize=11, transform=axes[0, 3].transAxes)
axes[0, 3].set_facecolor('#f0f0f0')
axes[0, 3].axis('off')
axes[1, 3].axis('off')

plt.suptitle('Example Patches by Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Clean up
slide.close()
cleanup_file(slide_path)
cleanup_file(xml_path)


## 7. Step 3 — Building the Training Dataset

With our labelling scheme defined, we need to extract hundreds of thousands of patches from hundreds of slides and organise them into a training dataset. This is the most computationally intensive preprocessing step.

### The Chunking Strategy

We can't hold all patches in memory simultaneously (400K patches × 224×224×3 ≈ **130 GB**). Instead, we save patches in **chunks** of ~1,000 patches each as compressed `.npz` files. Each chunk stores:
- `X`: Patch pixel arrays `(N, 224, 224, 3)`
- `y`: Class labels `(N,)`
- `slides`: Source slide IDs `(N,)` — critical for preventing data leakage
- `coords`: Original coordinates `(N, 2)` — for visualisation and debugging

### Stain Normalisation

H&E staining varies between labs, technicians, and even between slides from the same lab. This variation is irrelevant to the diagnostic task but can confuse a model. We apply **Macenko stain normalisation** during dataset generation: every patch is colour-adjusted to match a reference image, reducing spurious variation while preserving diagnostic features.

> **Dataset generation takes ~6–8 hours** on Colab and only needs to be run once. The generated dataset is saved to Google Drive for reuse. We provide the generator code below but skip execution — the pre-generated dataset is used for all subsequent steps.


In [ ]:
# === DATASET GENERATION (run once, then skip) ===
# This cell is provided for reference. The pre-generated dataset
# is loaded in the next section.

# from src.data.generator import generate_dataset, generate_test_dataset
#
# # Training dataset: ~100K patches per class
# generate_dataset(
#     class_targets={0: 100000, 1: 100000, 2: 100000, 3: 100000},
#     save_path='./data/camelyon16_4class_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )
#
# # Test dataset: ~25K patches per class
# generate_test_dataset(
#     class_targets={0: 25000, 1: 25000, 2: 25000, 3: 25000},
#     save_path='./data/camelyon16_test_stain_normalised',
#     stain_normalise=True,
#     reference_image_path='./data/reference_patch.png'
# )

print("Dataset generation code shown above (pre-generated dataset used below)")


In [ ]:
# Dataset paths
TRAIN_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_4class_stain_normalised'
TEST_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_test_stain_normalised'

# Verify the dataset
import os
from pathlib import Path

class_names = {
    0: 'normal_from_normal',
    1: 'normal_from_tumor',
    2: 'boundary_tumor',
    3: 'pure_tumor'
}

print("=== Training Dataset ===")
total_train = 0
for class_id, name in class_names.items():
    class_dir = Path(TRAIN_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_train += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_train:,} patches")

print("\n=== Test Dataset ===")
total_test = 0
for class_id, name in class_names.items():
    class_dir = Path(TEST_PATH) / name
    chunks = list(class_dir.glob('*.npz'))
    if chunks:
        sample = np.load(str(chunks[0]))
        n_per_chunk = len(sample['X'])
        sample.close()
        total = len(chunks) * n_per_chunk
        total_test += total
        print(f"  {name}: {len(chunks)} chunks (~{total:,} patches)")

print(f"  Total: ~{total_test:,} patches")

In [ ]:
# Inspect a single chunk
sample_chunk = np.load(
    str(next(Path(TRAIN_PATH, 'pure_tumor').glob('*.npz')))
)
print("Chunk contents:")
for key in sample_chunk.files:
    arr = sample_chunk[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

# Show some patches from this chunk
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    patch = sample_chunk['X'][i]
    # Scale to [0,1] for display if needed
    if patch.max() > 1.5:
        patch = patch / 255.0
    ax.imshow(np.clip(patch, 0, 1))
    ax.axis('off')
    ax.set_title(f"Slide: {sample_chunk['slides'][i]}", fontsize=8)

plt.suptitle('Sample Patches from a Pure Tumor Chunk', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
sample_chunk.close()


## 8. Step 4 — The Training Pipeline

With our chunked dataset ready, we need a training pipeline that can:
1. Stream patches from chunks without loading everything into memory
2. Remap our 4-class labels to binary labels for each experiment
3. Ensure **no slide leakage** between training and validation sets
4. Handle class balancing

### Memory Management: The Key Engineering Challenge

This was the single hardest engineering problem in the project. Loading 400K patches into memory would require ~130 GB of RAM — far more than Colab provides (even with Pro). Our solution uses TensorFlow's `tf.data` pipeline:

- **Training**: `tf.data.Dataset.interleave()` reads from multiple chunk files in parallel, yielding patches on-the-fly without holding all data in memory
- **Validation**: A pre-loaded, class-balanced subset cached in memory for stable metrics across epochs
- **Critical lesson**: We initially used `tf.data.AUTOTUNE` for parallel loading, which spawned too many workers and caused memory leaks. Setting explicit limits (`num_parallel_calls=2`, `prefetch(2)`) solved the issue.

### Preventing Slide Leakage

If patches from the same slide appear in both training and validation, the model can memorise slide-specific features (staining artefacts, tissue preparation) rather than learning genuine biological signals. Our pipeline splits at the **chunk level** and then verifies zero slide overlap using the `slides` array stored in each chunk.


### Under the Hood: Streaming Chunks with `tf.data`

The training pipeline must feed ~400K patches to the model without loading them all into memory. Here's the core idea — each chunk file is read on demand using `tf.data.interleave`:

```python
# Simplified version of our chunk reading logic:

def read_chunk(file_path, label):
    with np.load(file_path, mmap_mode="r") as data:
        X = data['X']                     # Memory-mapped, not loaded yet
        idx = np.random.choice(len(X), max_patches, replace=False)
        patches = X[idx].astype(np.float32)  # Only NOW loaded into RAM

        # Normalise to [0, 1]
        if patches.max() > 1.5:
            patches /= 255.0
        patches = np.clip(patches, 0.0, 1.0)

        labels = np.full(len(patches), label, dtype=np.int32)
        return patches, labels

# tf.data.interleave reads from multiple chunks simultaneously,
# yielding a stream of patches without holding everything in memory:
dataset = file_dataset.interleave(
    read_chunk,
    cycle_length=4,           # Read 4 chunks at once
    num_parallel_calls=2,     # CRITICAL: not AUTOTUNE (causes memory leaks)
    deterministic=False       # Allow out-of-order for speed
)
```

**Class balancing** is enforced at the batch level: we create separate streams for each class, batch half from each, then concatenate. This guarantees exactly 50/50 class balance in every batch:

```python
normal_ds = create_class_stream(normal_chunks, label=0)
tumor_ds  = create_class_stream(tumor_chunks,  label=1)

# Each batch: 16 normal + 16 tumor = 32 balanced samples
balanced = tf.data.Dataset.zip((
    normal_ds.batch(16),
    tumor_ds.batch(16)
)).map(lambda n, t: concat(n, t))
```

**Slide leakage prevention**: after splitting chunks into train/val, we read the `slides` array from every chunk and verify zero overlap:

```python
train_slides = collect_slide_ids(train_chunks)
val_slides   = collect_slide_ids(val_chunks)
assert len(train_slides & val_slides) == 0, "Slide leakage detected!"
```


In [ ]:
# The binary experiment framework
# Our training module supports 5 experiments with different class combinations:

experiments = {
    1: ("Normal vs Any Tumor",     "{0: [normal_from_normal], 1: [all 3 tumor classes]}"),
    2: ("Normal vs Pure Tumor",    "{0: [normal_from_normal], 1: [pure_tumor]}"),
    3: ("Slide Context Detection", "{0: [normal_from_normal], 1: [normal_from_tumor]}"),
    4: ("Normal vs Actual Tumor",  "{0: [normal_from_normal], 1: [boundary + pure_tumor]}"),
    5: ("Normal vs Boundary",      "{0: [normal_from_normal], 1: [boundary_tumor]}"),
}

print("Binary Classification Experiments")
print("=" * 70)
for exp_id, (name, mapping) in experiments.items():
    difficulty = {1: "Medium", 2: "Easy", 3: "Very Hard", 4: "Medium", 5: "Hard"}[exp_id]
    print(f"  Exp {exp_id}: {name:<30s} [{difficulty}]")
    print(f"          {mapping}")


## 9. Step 5 — Model Architecture

We deliberately keep our CNN architecture simple and interpretable. The goal is to demonstrate the end-to-end pipeline, not to push state-of-the-art accuracy (the CAMELYON16 winners used GoogLeNet with 6M+ parameters and millions of training patches).

Our primary model, `subtle`, uses four convolutional blocks with batch normalisation and dropout:

```
Input (224×224×3)
  → Conv2D(32, 3×3, stride=1) + BN + ReLU + Dropout(0.2)   [224×224]
  → Conv2D(64, 3×3, stride=2) + BN + ReLU + Dropout(0.3)   [112×112]
  → Conv2D(128, 3×3, stride=2) + BN + ReLU + Dropout(0.3)  [56×56]
  → Conv2D(256, 3×3, stride=2) + BN + ReLU + Dropout(0.4)  [28×28]
  → GlobalAveragePooling2D
  → Dropout(0.5)
  → Dense(1, sigmoid)

Total: ~390K parameters
```

Key design choices:
- **Small 3×3 kernels**: capture fine-grained tissue details (important for subtle differences)
- **Stride-based downsampling** instead of max pooling: preserves more spatial information
- **Global Average Pooling** instead of Flatten: dramatically reduces parameters and acts as a regulariser
- **Heavy dropout**: prevents overfitting on a relatively small dataset


In [ ]:
import tensorflow as tf
from tensorflow import keras
from src.models.architectures import get_model

# Build and inspect the model
model = get_model('subtle')
model.summary()


## 10. Step 6 — Training

We train with the following hyperparameters, chosen through experimentation:
- **Optimiser**: Adam with learning rate `1e-5` (very low — prevents training instability)
- **Gradient clipping**: `clipnorm=1.0` (prevents exploding gradients that cause wild validation oscillations)
- **Callbacks**: ModelCheckpoint (save best val_loss), ReduceLROnPlateau (halve LR after 3 stagnant epochs), EarlyStopping (stop after 3 epochs without improvement)

> **Note**: Training requires ~6 minutes per epoch on a T4 GPU. The cells below execute the full training runs. If you want to skip training, pre-trained models can be loaded from the `models/` directory.


In [ ]:
# Configure training
from src.models import run_binary_experiment
from config import DEFAULT_CONFIG

DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

# Uses TRAIN_PATH defined in Section 7 above
TRAIN_DATASET_PATH = TRAIN_PATH

# ============================================================
# EXPERIMENT 2: Normal vs Pure Tumor (sanity check — should be easy)
# ============================================================
print("=" * 60)
print("EXPERIMENT 2: Normal vs Pure Tumor")
print("=" * 60)

exp2_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=2,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp2_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp2_results['results']['auc']:.3f}")


In [ ]:
# ============================================================
# EXPERIMENT 5: Normal vs Boundary (harder)
# ============================================================
print("=" * 60)
print("EXPERIMENT 5: Normal vs Boundary Tumor")
print("=" * 60)

exp5_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=5,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp5_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp5_results['results']['auc']:.3f}")


In [ ]:
# ============================================================
# EXPERIMENT 3: Slide Context Detection (field cancerization)
# ============================================================
print("=" * 60)
print("EXPERIMENT 3: Slide Context Detection (Field Cancerization)")
print("=" * 60)

exp3_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=3,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp3_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp3_results['results']['auc']:.3f}")


## 11. Step 7 — Test Set Evaluation

Validation metrics are computed on held-out chunks from the same pool of training slides. The true test of generalisation is the **held-out test set**, which uses entirely different slides.

### A Critical Bug We Fixed

During development, we discovered that our test evaluation function was feeding **raw, unscaled pixel data** (0–255) to models trained on normalised data (0–1). Validation AUC was 0.93 but test AUC collapsed to 0.53 — essentially random chance.

The fix was simple: apply the same `[0, 1]` scaling and optional per-patch normalisation in the test evaluation path. The lesson was painful but valuable: **preprocessing consistency between training and inference is non-negotiable**, and test evaluation should always be your first sanity check.


In [ ]:
# Test set evaluation
from src.models import evaluate_on_test_set, load_model_metadata

# Uses TEST_PATH defined in Section 7 above

# Load models and metadata
experiments_to_eval = {
    'exp2': {
        'name': 'Normal vs Pure Tumor',
        'model_path': './models/normal_vs_pure_tumor.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['pure_tumor']},
        'results': exp2_results
    },
    'exp5': {
        'name': 'Normal vs Boundary',
        'model_path': './models/normal_vs_boundary.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['boundary_tumor']},
        'results': exp5_results
    },
    'exp3': {
        'name': 'Slide Context Detection',
        'model_path': './models/slide_context_detection.keras',
        'mapping': {0: ['normal_from_normal'], 1: ['normal_from_tumor']},
        'results': exp3_results
    }
}

test_results = {}
for key, exp in experiments_to_eval.items():
    print(f"\n{'='*60}")
    print(f"TEST: {exp['name']}")
    print(f"{'='*60}")

    model = keras.models.load_model(exp['model_path'])
    meta = load_model_metadata(exp['model_path'])
    normalise = meta.get('normalise_patches', False)

    result = evaluate_on_test_set(
        model, TEST_PATH, exp['mapping'], key,
        threshold=meta['threshold'],
        normalise=normalise
    )
    test_results[key] = result

    print(f"Val AUC:  {exp['results']['results']['auc']:.3f}")
    print(f"Test AUC: {result['auc']:.3f}")
    print(f"Gap:      {exp['results']['results']['auc'] - result['auc']:.3f}")
    print(result['report'])


## 12. Results

Let's compare validation and test performance across all experiments.


In [ ]:
# Summary comparison chart
exp_names = ['Exp 2: Normal vs\nPure Tumor', 'Exp 5: Normal vs\nBoundary', 'Exp 3: Slide\nContext']
exp_keys = ['exp2', 'exp5', 'exp3']

val_aucs = [experiments_to_eval[k]['results']['results']['auc'] for k in exp_keys]
test_aucs = [test_results[k]['auc'] for k in exp_keys]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(exp_names))
width = 0.35

# AUC comparison
bars1 = axes[0].bar(x - width/2, val_aucs, width, label='Validation', color='steelblue')
bars2 = axes[0].bar(x + width/2, test_aucs, width, label='Test', color='darkorange')
axes[0].set_ylabel('AUC', fontsize=12)
axes[0].set_title('Validation vs Test AUC', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_names, fontsize=10)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0.4, 1.0)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
for i, (v, t) in enumerate(zip(val_aucs, test_aucs)):
    axes[0].text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i + width/2, t + 0.02, f'{t:.3f}', ha='center', fontsize=9, fontweight='bold')

# Val-Test gap
gaps = [v - t for v, t in zip(val_aucs, test_aucs)]
colours = ['forestgreen' if g < 0.05 else 'darkorange' if g < 0.1 else 'firebrick' for g in gaps]
axes[1].bar(x, gaps, color=colours, width=0.5)
axes[1].set_ylabel('AUC Gap (Val - Test)', fontsize=12)
axes[1].set_title('Generalisation Gap', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_names, fontsize=10)
axes[1].axhline(y=0, color='black', linewidth=0.5)
for i, g in enumerate(gaps):
    axes[1].text(i, g + 0.005, f'{g:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 13. Investigating the Field Cancerization Hypothesis

The most scientifically interesting result is Experiment 3: can a model detect whether normal-looking tissue came from a slide that also contains a tumor?

Our initial validation results suggested a weak but detectable signal (AUC ~0.64). However, evaluation on the held-out test set showed performance near random chance (AUC ~0.54). To ensure this wasn't a model capacity issue, we tested four architectures of increasing complexity:

| Architecture | Val AUC | Test AUC | Trainable Params |
|---|---|---|---|
| `subtle` (custom CNN) | 0.585 | 0.543 | ~390K |
| `attention` (spatial attention) | 0.610 | 0.467 | ~390K |
| `transfer` (frozen MobileNetV2) | 0.504 | — | ~1.3K |
| `transfer_finetune` (fine-tuned MobileNetV2) | 0.585 | — | ~700K |

The consistency across architectures is telling: the problem isn't model capacity. The weak validation signal likely reflects **slide-level confounds** (staining batch effects, tissue preparation differences) rather than genuine field cancerization. These confounds are shared between training and validation slides (from the same pool) but don't transfer to the test slides.

### What Would It Take?

Properly testing this hypothesis would require approaches beyond patch-level classification:
- **Multi-instance learning**: aggregate evidence across many patches per slide, rather than classifying patches independently
- **Pathology foundation models**: use feature extractors pre-trained on millions of pathology images (e.g., UNI, CONCH) rather than ImageNet
- **Slide-level prediction**: treat each slide as a single example, using all its patches collectively

This is itself a valuable finding: a clean negative result that establishes what *doesn't* work and points toward what might.


## 14. Lessons Learned

### Technical Lessons

1. **Preprocessing consistency is everything.** Our most dramatic bug was a mismatch between training and test preprocessing: models saw `[0, 1]` scaled data during training but raw `[0, 255]` data during evaluation. Validation AUC was 0.93; test AUC was 0.53. Always have a single source of truth for preprocessing transformations.

2. **Memory management dominates development time.** Whole slide images, patch datasets, and TensorFlow data pipelines all compete for RAM. The most impactful fixes weren't algorithmic — they were setting `num_parallel_calls=2` instead of `AUTOTUNE`, limiting validation set size, and using generators instead of loading data into memory.

3. **Gradient clipping matters more than architecture changes.** Our initial training runs showed wild validation accuracy oscillations (±20% between epochs). Gradient clipping (`clipnorm=1.0`) combined with a low learning rate (`1e-5`) was more effective than any architectural modification.

4. **Start with the easiest experiment.** We used Normal vs Pure Tumor (the visually obvious case) to validate the entire train→evaluate→test pipeline before investing GPU time on harder tasks. This caught the preprocessing bug early.

### Scientific Lessons

5. **Negative results are results.** Our field cancerization experiment didn't find a generalisable signal, despite trying four architectures. This is informative: it constrains what's possible with patch-level H&E classification and motivates alternative approaches.

6. **Validation ≠ generalisation.** Even with no slide leakage, validation and test performance can diverge significantly when the task is hard. Validation slides share distributional properties with training slides; test slides may not.

7. **Know when to stop.** When four architectures all converge on the same weak result, the bottleneck is likely the data or the task, not the model. Throwing more complexity at a data-limited problem is unlikely to help.


## 15. Conclusion

We built a complete pipeline for automated tumor detection in histopathology images: from gigapixel whole slide images through tissue detection, patch extraction, stain normalisation, and CNN-based classification. Our models achieve **0.87–0.89 test AUC** for detecting tumor tissue — a useful screening tool, though short of the 0.925 achieved by the CAMELYON16 winners with larger models and more data.

The field cancerization hypothesis — that normal tissue near tumors carries detectable molecular changes — remains unresolved by our patch-level approach. The signal we observed in validation did not survive the test set, suggesting that more sophisticated methods (multi-instance learning, pathology foundation models) are needed to investigate this further.

The full codebase, including all modules, configurations, and notebooks, is available on [GitHub](https://github.com/your-username/camelyon16-pathology).

---

*This work was conducted as part of a computational pathology project at [Company Name]. The CAMELYON16 dataset is publicly available through the [Grand Challenge platform](https://camelyon16.grand-challenge.org/).*
